## Предобработка данных по ДТП

Цель: выгрузить из БД полученные из API ГИБДД РФ данные и провести их предварительную очистку, обработку, форматирование для дальнейшей работы с данными и их анализа.
Очищенные данные вновь загрузить в БД.

In [1]:
# импорт библиотек
import time
from datetime import datetime, timedelta
from typing import List, Dict, Optional

import pandas as pd
import numpy as np

import sqlalchemy
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os


In [2]:
# Подключение к БД Supabase
load_dotenv()

# Параметры подключения
user = os.getenv('user')
password = os.getenv('password')
host = os.getenv('host')
port = '6543'
dbname = os.getenv('dbname')

# Адрес подключения
DATABASE_URL = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?sslmode=require"

# Формируем движок
engine = create_engine(DATABASE_URL) 

try:
    with engine.connect() as connection:
        print(f'Подключение к базе успешно')
except Exception as e:
        print(f'Ошибка при подключении к базе: {e}')

Подключение к базе успешно


In [3]:
query1 = '''
   SELECT *
   FROM main_df;
  '''
main_df = pd.read_sql_query(query1, con=engine)

main_df.info()
main_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20224 entries, 0 to 20223
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   index        20224 non-null  int64 
 1   KartId       20224 non-null  int64 
 2   rowNum       20224 non-null  int64 
 3   date         20224 non-null  object
 4   Time         20224 non-null  object
 5   District     20224 non-null  object
 6   DTP_V        20224 non-null  object
 7   POG          20224 non-null  int64 
 8   RAN          20224 non-null  int64 
 9   K_TS         20224 non-null  int64 
 10  K_UCH        20224 non-null  int64 
 11  emtp_number  20224 non-null  object
dtypes: int64(7), object(5)
memory usage: 1.9+ MB


,index,KartId,rowNum,date,Time,District,DTP_V,POG,RAN,K_TS,K_UCH,emtp_number
0,0,161237026,1,31.01.2015,15:30,Октябрьский,Наезд на стоящее ТС,0,1,2,2,
1,1,161340310,2,31.01.2015,14:00,Чкаловский,Наезд на пешехода,0,1,1,2,
2,2,161238027,3,31.01.2015,03:35,Верх-Исетский,Наезд на препятствие,0,1,1,1,
3,3,171517613,4,31.01.2015,09:45,Кировский,Наезд на пешехода,0,1,1,2,
4,4,160935023,5,30.01.2015,02:25,Верх-Исетский,Столкновение,0,1,2,3,


In [4]:
main_clean = main_df.drop(columns = ['index','rowNum','emtp_number']) # удалим лишние столбцы

# мнеяем формат даты
main_clean['date'] = pd.to_datetime(main_clean['date'], format='%d.%m.%Y')
# main['Time'] = pd.to_datetime(main['Time'], format='%H:%M').dt.time

# добавляем отдельные столбы с годов, месяцем, днем
main_clean['год'] = main_clean['date'].dt.year
main_clean['месяц'] = main_clean['date'].dt.month
main_clean['день'] = main_clean['date'].dt.day

#переименуем столбцы
main_clean = main_clean.rename(columns={
                            'District': 'район', 'DTP_V': 'вид_ДТП',
                            'POG': 'погибло', 'RAN': 'ранено',
                             'K_TS': 'количество ТС', 'K_UCH': 'количество участников',
                                 'date':'дата', 'Time':'время'
                            })
main_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20224 entries, 0 to 20223
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   KartId                 20224 non-null  int64         
 1   дата                   20224 non-null  datetime64[ns]
 2   время                  20224 non-null  object        
 3   район                  20224 non-null  object        
 4   вид_ДТП                20224 non-null  object        
 5   погибло                20224 non-null  int64         
 6   ранено                 20224 non-null  int64         
 7   количество ТС          20224 non-null  int64         
 8   количество участников  20224 non-null  int64         
 9   год                    20224 non-null  int32         
 10  месяц                  20224 non-null  int32         
 11  день                   20224 non-null  int32         
dtypes: datetime64[ns](1), int32(3), int64(5), object(3)
memory u

In [5]:
# выгружаем только нужные для анализа столбцы
query2 = '''
    SELECT "KartId","n_p", "k_ul", "dor_z", "s_pch", "osv",  "ndu", "sdor", "s_pog"
    from info_dtp_df;
  '''
info_dtp = pd.read_sql_query(query2, con=engine)
info_dtp.head()

,KartId,n_p,k_ul,dor_z,s_pch,osv,ndu,sdor,s_pog
0,161237026,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Ясно
1,161340310,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Ясно
2,161238027,г Екатеринбург,Магистральные улицы районного значения,Не указано,Обработанное противогололедными материалами,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Пасмурно
3,171517613,г Екатеринбург,Магистральные улицы районного значения,Не указано,Гололедица,"В темное время суток, освещение включено",Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Ясно
4,160935023,г Екатеринбург,Магистральные улицы районного значения,Не указано,Сухое,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Пасмурно


In [6]:
info_dtp_clean = info_dtp.rename(columns={'k_ul': 'категория_улиц', 'dor_z': 'категория_дорог',
                            's_pch': 'покрытие', 'osv': 'освещенность',
                            'ndu': 'недостатки_дороги ', 'sdor': 'схема_расположения',
                             's_pog': 'погодные условия','n_p':'город'
                            })
info_dtp_clean.head()


,KartId,город,категория_улиц,категория_дорог,покрытие,освещенность,недостатки_дороги,схема_расположения,погодные условия
0,161237026,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Ясно
1,161340310,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Ясно
2,161238027,г Екатеринбург,Магистральные улицы районного значения,Не указано,Обработанное противогололедными материалами,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Пасмурно
3,171517613,г Екатеринбург,Магистральные улицы районного значения,Не указано,Гололедица,"В темное время суток, освещение включено",Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Ясно
4,160935023,г Екатеринбург,Магистральные улицы районного значения,Не указано,Сухое,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Пасмурно


In [7]:
# выделим город и KartId в отдельную таблицу
info_city = info_dtp_clean[['KartId','город']]

info_city['город'] = info_city['город'].str.replace(r'^г\.?\s*', '', regex=True)
info_city

C:\Users\Sandy\AppData\Local\Temp\ipykernel_12608\1704730086.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  info_city['город'] = info_city['город'].str.replace(r'^г\.?\s*', '', regex=True)


,KartId,город
0,161237026,Екатеринбург
1,161340310,Екатеринбург
2,161238027,Екатеринбург
3,171517613,Екатеринбург
4,160935023,Екатеринбург
...,...,...
20219,225219802,Владивосток
20220,225222817,Владивосток
20221,225228096,Владивосток
20222,225219810,Владивосток


In [8]:
info_dtp_clean['город'] = info_dtp_clean['город'].str.replace(r'^г\.?\s*', '', regex=True)
info_dtp_clean

,KartId,город,категория_улиц,категория_дорог,покрытие,освещенность,недостатки_дороги,схема_расположения,погодные условия
0,161237026,Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Ясно
1,161340310,Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Ясно
2,161238027,Екатеринбург,Магистральные улицы районного значения,Не указано,Обработанное противогололедными материалами,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Пасмурно
3,171517613,Екатеринбург,Магистральные улицы районного значения,Не указано,Гололедица,"В темное время суток, освещение включено",Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Ясно
4,160935023,Екатеринбург,Магистральные улицы районного значения,Не указано,Сухое,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Пасмурно
...,...,...,...,...,...,...,...,...,...
20219,225219802,Владивосток,Улицы и дороги местного значения научно-произв...,"Местного значения (дорога местного значения, в...",Со снежным накатом,Светлое время суток,Недостатки зимнего содержания,Нерегулируемый перекрёсток равнозначных улиц (...,Ясно
20220,225222817,Владивосток,Магистральные улицы общегородского значения,"Местного значения (дорога местного значения, в...",Мокрое,Светлое время суток,Не установлены,Нерегулируемый перекрёсток неравнозначных улиц...,Ясно
20221,225228096,Владивосток,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",Мокрое,"В темное время суток, освещение включено",Не установлены,Перегон (нет объектов на месте ДТП),Ясно
20222,225219810,Владивосток,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",Мокрое,Светлое время суток,Не установлены,Нерегулируемый перекрёсток неравнозначных улиц...,Ясно


In [9]:
# добавим дату и время из первой тоблицы
info_dtp_clean = pd.merge(info_dtp_clean, main_clean[['KartId','дата', 'время', 'год', 'месяц']], on ='KartId', how='inner')
info_dtp_clean.head()

,KartId,город,категория_улиц,категория_дорог,покрытие,освещенность,недостатки_дороги,схема_расположения,погодные условия,дата,время,год,месяц
0,161237026,Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Ясно,2015-01-31,15:30,2015,1
1,161340310,Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Ясно,2015-01-31,14:00,2015,1
2,161238027,Екатеринбург,Магистральные улицы районного значения,Не указано,Обработанное противогололедными материалами,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Пасмурно,2015-01-31,03:35,2015,1
3,171517613,Екатеринбург,Магистральные улицы районного значения,Не указано,Гололедица,"В темное время суток, освещение включено",Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Ясно,2015-01-31,09:45,2015,1
4,160935023,Екатеринбург,Магистральные улицы районного значения,Не указано,Сухое,"В темное время суток, освещение включено","Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Пасмурно,2015-01-30,02:25,2015,1


In [10]:
query3 = '''
    SELECT "KartId","t_ts", "marka_ts","g_v", "m_pov","t_n"
                           
    from ts_df;
  '''
ts = pd.read_sql_query(query3, con=engine)
ts.head()

,KartId,t_ts,marka_ts,g_v,m_pov,t_n
0,161237026,"D-класс (средний) до 4,6 м",ВАЗ,2013,Передний правый угол | Передний левый угол,Технические неисправности отсутствуют
1,161237026,"D-класс (средний) до 4,6 м",OPEL,2007,Задний правый угол | Задний левый угол,Технические неисправности отсутствуют
2,161340310,"D-класс (средний) до 4,6 м",FORD,2007,,Технические неисправности отсутствуют
3,161238027,Минивэны и универсалы повышенной вместимости,MAZDA,2010,Крыша | Полная деформация кузова | Смещение дв...,Технические неисправности отсутствуют
4,171517613,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,2008,Передний правый угол,Технические неисправности отсутствуют


In [11]:
ts_clean = ts.rename(columns={'t_ts': 'тип_тс', 'marka_ts': 'марка_тс',
                            'g_v': 'год_выпуска', 'm_pov': 'повреждения',
                            't_n': 'наличие_тех.неисправностей'
                            })

ts_clean.head()

,KartId,тип_тс,марка_тс,год_выпуска,повреждения,наличие_тех.неисправностей
0,161237026,"D-класс (средний) до 4,6 м",ВАЗ,2013,Передний правый угол | Передний левый угол,Технические неисправности отсутствуют
1,161237026,"D-класс (средний) до 4,6 м",OPEL,2007,Задний правый угол | Задний левый угол,Технические неисправности отсутствуют
2,161340310,"D-класс (средний) до 4,6 м",FORD,2007,,Технические неисправности отсутствуют
3,161238027,Минивэны и универсалы повышенной вместимости,MAZDA,2010,Крыша | Полная деформация кузова | Смещение дв...,Технические неисправности отсутствуют
4,171517613,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,2008,Передний правый угол,Технические неисправности отсутствуют


In [12]:
query4 = '''
    SELECT "KartId","K_UCH", "S_T","POL", "V_ST","ALCO", "SAFETY_BELT", "NPDD","SOP_NPDD"
    from uch_data_df;
  '''
uch = pd.read_sql_query(query4, con=engine)
uch.head()

,KartId,K_UCH,S_T,POL,V_ST,ALCO,SAFETY_BELT,NPDD,SOP_NPDD
0,161237026,Водитель,Не пострадал,Мужской,17,,Да,Несоответствие скорости конкретным условиям дв...,Нет нарушений
1,161237026,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,7,,Да,Нет нарушений,Нет нарушений
2,161340310,Водитель,Не пострадал,Мужской,10,,Да,"Несоблюдение условий, разрешающих движение тра...",Оставление места ДТП
3,161238027,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,2,15,Да,Другие нарушения ПДД водителем,Управление ТС в состоянии алкогольного опьянения
4,171517613,Водитель,Не пострадал,Мужской,2,,Да,Нет нарушений,Нет нарушений


In [13]:
uch_clean = uch.rename(columns={'K_UCH': 'категория_пострадавшего', 'S_T': 'степень_тяжести_пострадавшего',
                            'V_ST': 'стаж_вождения', 'ALCO': 'наличие_алкоголя',
                            'SAFETY_BELT': 'пристегнут_ремень_безопасности',
                             'NPDD': 'нарушение_пдд', 'SOP_NPDD': 'дополнительные_нарушения',
                              'POL': 'пол'
                            })

uch_clean.head()

,KartId,категория_пострадавшего,степень_тяжести_пострадавшего,пол,стаж_вождения,наличие_алкоголя,пристегнут_ремень_безопасности,нарушение_пдд,дополнительные_нарушения
0,161237026,Водитель,Не пострадал,Мужской,17,,Да,Несоответствие скорости конкретным условиям дв...,Нет нарушений
1,161237026,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,7,,Да,Нет нарушений,Нет нарушений
2,161340310,Водитель,Не пострадал,Мужской,10,,Да,"Несоблюдение условий, разрешающих движение тра...",Оставление места ДТП
3,161238027,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,2,15,Да,Другие нарушения ПДД водителем,Управление ТС в состоянии алкогольного опьянения
4,171517613,Водитель,Не пострадал,Мужской,2,,Да,Нет нарушений,Нет нарушений


In [14]:
def add_city(df: pd.DataFrame):
    
    ''' добаваляет название города'''

    df = pd.merge(df,info_city, on='KartId', how ='outer')
    return df

In [15]:
# добавим название города в каждый датафрейм
main_clean = add_city(main_clean)
ts_clean = add_city(ts_clean)
uch_clean = add_city(uch_clean)

In [16]:
def replace_space(df: pd.DataFrame, columns: list, replacement='нет данных')-> pd.DataFrame:
    '''
    Заменяет пробелы, пропуски в столбах датафрейма на заданное значение 'нет данных'
    '''
    # Значения для замены
    missing_values = ['', ' ', 'NULL', 'null', 'NaN', 'nan']

    for column in columns:
       # Очищаем пробелы и заменяем указанные строки на NaN
        df[column] = df[column].astype(str).str.strip()
        df[column] = df[column].replace(missing_values, np.nan)
        # Заменяем все NaN на заданное значение
        df[column] = df[column].fillna(replacement)
    return df

In [17]:
columns_to_fix = ['марка_тс', 'год_выпуска', 'повреждения']  
ts_clean = replace_space(ts_clean, columns_to_fix, replacement='нет данных')

ts_clean.head()

,KartId,тип_тс,марка_тс,год_выпуска,повреждения,наличие_тех.неисправностей,город
0,157114292,Минивэны и универсалы повышенной вместимости,VOLKSWAGEN,2010,Передний правый угол | Передний левый угол,Технические неисправности отсутствуют,Екатеринбург
1,157114292,"В-класс (малый) до 3,9 м",ВАЗ,2002,Задний левый угол | Задний левый бок (задняя д...,Технические неисправности отсутствуют,Екатеринбург
2,157122648,"В-класс (малый) до 3,9 м",MAZDA,2008,Передний правый угол,Технические неисправности отсутствуют,Екатеринбург
3,157156173,"D-класс (средний) до 4,6 м",ВАЗ,2008,Передний правый угол | Задний левый угол | Пер...,Технические неисправности отсутствуют,Екатеринбург
4,157223255,"В-класс (малый) до 3,9 м",TOYOTA,1994,Передний правый угол | Передний левый бок (пер...,Технические неисправности отсутствуют,Владивосток


In [18]:
uch_clean = replace_space(uch_clean, ['стаж_вождения', 'степень_тяжести_пострадавшего', 'наличие_алкоголя', 'пристегнут_ремень_безопасности', 'нарушение_пдд'])
uch_clean.head()

,KartId,категория_пострадавшего,степень_тяжести_пострадавшего,пол,стаж_вождения,наличие_алкоголя,пристегнут_ремень_безопасности,нарушение_пдд,дополнительные_нарушения,город
0,157114292,Водитель,Не пострадал,Мужской,16,нет данных,Да,Несоответствие скорости конкретным условиям дв...,Несоблюдение требований ОСАГО,Екатеринбург
1,157114292,Водитель,Не пострадал,Мужской,5,нет данных,Да,Нет нарушений,Нет нарушений,Екатеринбург
2,157114292,Пассажир,"Раненый, находящийся (находившийся) на стацион...",Женский,нет данных,нет данных,Да,Нет нарушений,Нет нарушений,Екатеринбург
3,157122648,Водитель,Не пострадал,Мужской,10,нет данных,Да,Нет нарушений,Несоблюдение требований ОСАГО,Екатеринбург
4,157156173,Водитель,"Раненый, находящийся (находившийся) на стацион...",Мужской,4,нет данных,Да,Несоответствие скорости конкретным условиям дв...,Нет нарушений,Екатеринбург


### Загрузка очищенных данных в базу

In [19]:
# загружаем очищенные датафреймы обратно в базу
try:
    main_clean.to_sql('main_clean', con=engine, if_exists='replace')
    info_dtp_clean.to_sql('info_dtp_clean', con=engine, if_exists='replace')
    ts_clean.to_sql('ts_clean', con=engine, if_exists='replace')
    uch_clean.to_sql('uch_clean', con=engine, if_exists='replace')
    print(f'Загрузка  успешна')
   
except Exception as e:
    print(f'Ошибка при выполнении скрипта: {e}')

Загрузка  успешна
